# 03 Inverse Operator

This notebook creates MNE inverse operators for selected recordings.

Inputs per recording:

- recording info input selected by the pipeline (`epochs`, cleaned raw, or evoked),
- forward solution from `01_forward_solution.ipynb`,
- noise covariance from `02_noise_covariance.ipynb`.

## Setup

In [ ]:
from __future__ import annotations

from pathlib import Path

import mne
import pandas as pd

from meeg_pipeline.config import load_config
from meeg_pipeline.source_modeling import (
    inverse_operator_input_overview_to_dataframe,
    inverse_operator_qc_to_dataframe,
    inverse_operator_results_to_dataframe,
    source_forward_config_to_dataframe,
    write_inverse_operators_for_recordings,
)
from meeg_pipeline.workflow import (
    existing_output_policy_for_step,
    iter_recordings,
    selected_recordings_to_dataframe,
    should_overwrite,
)


def find_project_root(start: Path | None = None) -> Path:
    """Find project root by searching upward for configs/local.yaml."""
    start = Path.cwd() if start is None else Path(start).resolve()

    for candidate in [start, *start.parents]:
        if (candidate / "configs" / "local.yaml").exists():
            return candidate

    raise FileNotFoundError(
        "Could not find project root by searching for configs/local.yaml "
        f"above {start}"
    )


PROJECT_ROOT = find_project_root()
CONFIG_PATH = PROJECT_ROOT / "configs" / "local.yaml"
config = load_config(CONFIG_PATH)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CONFIG_PATH:", CONFIG_PATH)

## MNE logging

In [ ]:
mne.set_log_level("WARNING")

## Selection

`iter_recordings(..., "all")` uses existing raw-BIDS recordings and excludes `sub-emptyroom` by default.

In [ ]:
SUBJECTS = "all"
SESSIONS = "all"
TASKS = "all"
RUNS = "all"

RUN_INVERSE_OPERATOR_QC = True
MAX_INVERSE_OPERATOR_QC_ROWS = None  # None = all, or e.g. 3 for a quick test

selected_recordings = list(
    iter_recordings(
        config,
        subjects=SUBJECTS,
        sessions=SESSIONS,
        tasks=TASKS,
        runs=RUNS,
    )
)

selected_recordings_to_dataframe(selected_recordings)

## Overwrite policy

Default: skip existing inverse operators.

Set `OVERWRITE_STEPS = ["inverse_operator"]` to recompute existing inverse operators.

In [ ]:
OVERWRITE_STEPS = []

pd.DataFrame(
    [
        {
            "step": "inverse_operator",
            "overwrite": should_overwrite("inverse_operator", OVERWRITE_STEPS),
            "policy": existing_output_policy_for_step(
                "inverse_operator",
                OVERWRITE_STEPS,
            ),
        }
    ]
)

## Inverse-operator parameters

The inverse operator itself is method-agnostic; `INVERSE_METHOD` is stored in the filename so downstream source-estimate derivatives stay aligned with the configured method.

In [ ]:
SOURCE_SPACING = config.source.spacing
NOISE_COV_MODE = config.source.noise_cov.mode
INVERSE_METHOD = config.source.inverse.method

LOOSE = 0.2
DEPTH = 0.8
RANK = None

pd.concat(
    [
        source_forward_config_to_dataframe(config),
        pd.DataFrame(
            [
                {
            "source_spacing": SOURCE_SPACING,
            "noise_cov_mode": NOISE_COV_MODE,
            "inverse_method": INVERSE_METHOD,
            "loose": LOOSE,
            "depth": DEPTH,
            "rank": RANK,
                }
            ]
        ),
    ],
    axis=1,
)

## Input overview

This table checks whether all required files exist before attempting to write inverse operators.

In [ ]:
inverse_policy = existing_output_policy_for_step(
    "inverse_operator",
    OVERWRITE_STEPS,
)

inverse_overview = inverse_operator_input_overview_to_dataframe(
    config,
    selected_recordings,
    on_existing=inverse_policy,
    spacing=SOURCE_SPACING,
    noise_cov_mode=NOISE_COV_MODE,
    inverse_method=INVERSE_METHOD,
)

inverse_overview

## Status summary

In [ ]:
if inverse_overview.empty:
    pd.DataFrame()
else:
    (
        inverse_overview
        .groupby("status", dropna=False)
        .size()
        .reset_index(name="n_recordings")
        .sort_values(["status"])
    )

## Ready jobs

In [ ]:
ready_inverse_jobs = inverse_overview.query("status == 'ready'").copy()

columns = [
    "subject",
    "session",
    "task",
    "run",
    "fwd_desc",
    "fwd_meg",
    "fwd_eeg",
    "info_input_kind",
    "info_input",
    "fwd_path",
    "cov_path",
    "inverse_path",
]
existing_columns = [column for column in columns if column in ready_inverse_jobs.columns]
ready_inverse_jobs[existing_columns]

## Write inverse operators

This cell processes all selected recordings. Existing outputs are skipped unless `OVERWRITE_STEPS` contains `"inverse_operator"`. Missing inputs and per-recording failures are returned as status rows rather than stopping the whole batch.

In [ ]:
inverse_results = write_inverse_operators_for_recordings(
    config,
    selected_recordings,
    on_existing=inverse_policy,
    spacing=SOURCE_SPACING,
    noise_cov_mode=NOISE_COV_MODE,
    inverse_method=INVERSE_METHOD,
    loose=LOOSE,
    depth=DEPTH,
    rank=RANK,
    verbose=True,
)

inverse_results_df = inverse_operator_results_to_dataframe(inverse_results)
inverse_results_df

## Result summary

In [ ]:
if "inverse_results_df" not in globals() or inverse_results_df.empty:
    pd.DataFrame()
else:
    (
        inverse_results_df
        .groupby("status", dropna=False)
        .size()
        .reset_index(name="n_recordings")
        .sort_values(["status"])
    )

## Batch QC

Reads the written or existing inverse operators and returns a compact QC table. This is non-interactive and safe for batch runs.

In [ ]:
if RUN_INVERSE_OPERATOR_QC:
    if "inverse_results_df" not in globals() or inverse_results_df.empty:
        inverse_qc_status = pd.DataFrame(
            [{"status": "no_results", "message": "Run the write cell first."}]
        )
    else:
        candidate_results = inverse_results_df[
            inverse_results_df["status"].isin(["written", "skipped_existing", "exists"])
        ].copy()

        if candidate_results.empty:
            candidate_results = inverse_results_df.copy()

        inverse_qc_status = inverse_operator_qc_to_dataframe(
            candidate_results,
            max_rows=MAX_INVERSE_OPERATOR_QC_ROWS,
        )

    inverse_qc_status
else:
    print("Skipped inverse-operator QC.")

## Expected outputs

Inverse operators are written to:

```text
derivatives/meeg-pipeline/sub-*/meg/inverse/*_space-*_desc-*-inv.fif
```